# Little Lemon Analysis and Sales Report
Back-end: MySQL 8.4. Front-end: Python with MySQL Connector/Python.
All tasks use a connection taken from a connection pool, run through stored procedures, commit their changes, and return the connection to the pool.

## Task 1: Create a connection pool with 2 connections

In [ ]:
from mysql.connector import Error
from mysql.connector.pooling import MySQLConnectionPool

dbconfig = {
    "host": "localhost",  
    "port": 3306,  
    "user": "root",
    "password": "mihir", 
}

try:
    pool = MySQLConnectionPool(pool_name="pool_a", pool_size=2, **dbconfig)
    print("The connection pool is created with a name: ", pool.pool_name)
    print("The pool size is:", pool.pool_size)
except Error as er:
    print("Error message:", er.msg)

The connection pool is created with a name:  pool_a
The pool size is: 2


## Task 2: Create the database and tables, then populate them
Uses one connection from the pool. The connection is closed (returned to the pool) at the end.

In [9]:
connection = None
try:
    connection = pool.get_connection()
    cursor = connection.cursor()
    print("A connection is taken from the pool.")

    cursor.execute("DROP DATABASE IF EXISTS little_lemon_db")
    cursor.execute("CREATE DATABASE little_lemon_db")
    cursor.execute("USE little_lemon_db")
    print("Database little_lemon_db is created.")

    cursor.execute("""CREATE TABLE Employees (
        EmployeeID INT PRIMARY KEY AUTO_INCREMENT,
        Name VARCHAR(100) NOT NULL,
        Role VARCHAR(50) NOT NULL,
        Address VARCHAR(150),
        Contact_Number VARCHAR(20),
        Email VARCHAR(100),
        Annual_Salary INT)""")
    cursor.execute("""CREATE TABLE MenuItems (
        ItemID INT PRIMARY KEY AUTO_INCREMENT,
        ItemName VARCHAR(100) NOT NULL,
        Category VARCHAR(50) NOT NULL,
        Price DECIMAL(6,2) NOT NULL)""")
    cursor.execute("""CREATE TABLE Bookings (
        BookingID INT PRIMARY KEY AUTO_INCREMENT,
        TableNo INT NOT NULL,
        Guest_name VARCHAR(100) NOT NULL,
        BookingSlot TIME NOT NULL,
        EmployeeID INT NOT NULL,
        FOREIGN KEY (EmployeeID) REFERENCES Employees(EmployeeID))""")
    cursor.execute("""CREATE TABLE Orders (
        OrderID INT PRIMARY KEY AUTO_INCREMENT,
        BookingID INT NOT NULL,
        ItemID INT NOT NULL,
        Quantity INT NOT NULL,
        FOREIGN KEY (BookingID) REFERENCES Bookings(BookingID),
        FOREIGN KEY (ItemID) REFERENCES MenuItems(ItemID))""")
    print("Tables Employees, MenuItems, Bookings and Orders are created.")

    cursor.executemany("INSERT INTO Employees (Name, Role, Address, Contact_Number, Email, Annual_Salary) VALUES (%s,%s,%s,%s,%s,%s)", [
        ("Mario Gollini", "Manager", "724, Parsley Lane, Old Town, Chicago, IL", "351258074", "Mario.g@littlelemon.com", 70000),
        ("Adrian Gollini", "Assistant Manager", "334, Dill Square, Lincoln Park, Chicago, IL", "351474048", "Adrian.g@littlelemon.com", 65000),
        ("Giorgos Dioudis", "Head Chef", "879 Sage Street, West Loop, Chicago, IL", "351970582", "Giorgos.d@littlelemon.com", 50000),
        ("Fatma Kaya", "Assistant Chef", "132 Bay Lane, Chicago, IL", "351963569", "Fatma.k@littlelemon.com", 45000),
        ("Elena Salvai", "Head Waiter", "989 Thyme Square, EdgeWater, Chicago, IL", "351074198", "Elena.s@littlelemon.com", 40000),
        ("John Millar", "Receptionist", "245 Dill Square, Lincoln Park, Chicago, IL", "351584508", "John.m@littlelemon.com", 35000)])
    cursor.executemany("INSERT INTO MenuItems (ItemName, Category, Price) VALUES (%s,%s,%s)", [
        ("Greek Salad", "Starters", 12.50), ("Bruschetta", "Starters", 8.00), ("Grilled Fish", "Mains", 24.00),
        ("Pasta", "Mains", 18.50), ("Lemon Dessert", "Desserts", 7.50), ("Greek Yoghurt", "Desserts", 6.00)])
    cursor.executemany("INSERT INTO Bookings (TableNo, Guest_name, BookingSlot, EmployeeID) VALUES (%s,%s,%s,%s)", [
        (8, "Anna Iversen", "19:00:00", 1), (5, "Joakim Iversen", "19:00:00", 2), (12, "Vanessa McCarthy", "15:00:00", 3),
        (12, "Marcos Romero", "17:30:00", 4), (19, "Hiroki Yamane", "18:30:00", 5), (15, "Diana Pinto", "20:00:00", 6),
        (3, "Anees Java", "18:00:00", 6)])
    cursor.executemany("INSERT INTO Orders (BookingID, ItemID, Quantity) VALUES (%s,%s,%s)", [
        (1,1,2),(1,3,2),(1,5,2),(2,2,1),(2,4,1),(3,1,1),(3,4,2),(4,3,1),(4,6,1),(5,4,3),(5,5,3),(6,3,2),(6,1,2),(7,2,2),(7,4,1),(7,6,2)])

    connection.commit()   
    print("All tables are populated and the changes are committed.")
    for t in ("Employees", "MenuItems", "Bookings", "Orders"):
        cursor.execute(f"SELECT COUNT(*) FROM {t}")
        print(f"  {t}: {cursor.fetchone()[0]} rows")
except Error as er:
    print("Error code:", er.errno)
    print("Error message:", er.msg)
finally:
    if connection is not None and connection.is_connected():
        cursor.close()
        connection.close()
        print("The connection is returned to the pool.")

A connection is taken from the pool.
Database little_lemon_db is created.
Tables Employees, MenuItems, Bookings and Orders are created.
All tables are populated and the changes are committed.
  Employees: 6 rows
  MenuItems: 6 rows
  Bookings: 7 rows
  Orders: 16 rows
The connection is returned to the pool.


## Task 3: Create the stored procedures
Every later task calls one of these procedures.

In [ ]:
connection = None
try:
    connection = pool.get_connection()
    cursor = connection.cursor()
    cursor.execute("USE little_lemon_db")
    print("A connection is taken from the pool.")

    procs = {
    "GetBookingsWithStaff": """
        CREATE PROCEDURE GetBookingsWithStaff()
        BEGIN
            SELECT b.BookingSlot, b.Guest_name, e.Name, e.Role
            FROM Bookings b JOIN Employees e ON e.EmployeeID = b.EmployeeID
            ORDER BY b.BookingSlot;
        END""",
    "AddBooking": """
        CREATE PROCEDURE AddBooking(IN p_table INT, IN p_guest VARCHAR(100), IN p_slot TIME, IN p_emp INT)
        BEGIN
            INSERT INTO Bookings (TableNo, Guest_name, BookingSlot, EmployeeID) VALUES (p_table, p_guest, p_slot, p_emp);
            SELECT LAST_INSERT_ID() AS NewBookingID;
        END""",
    "UpdateBookingSlot": """
        CREATE PROCEDURE UpdateBookingSlot(IN p_id INT, IN p_slot TIME)
        BEGIN
            UPDATE Bookings SET BookingSlot = p_slot WHERE BookingID = p_id;
        END""",
    "CancelBooking": """
        CREATE PROCEDURE CancelBooking(IN p_id INT)
        BEGIN
            DELETE FROM Orders WHERE BookingID = p_id;
            DELETE FROM Bookings WHERE BookingID = p_id;
        END""",
    "SalesByItem": """
        CREATE PROCEDURE SalesByItem()
        BEGIN
            SELECT m.ItemName, m.Category, SUM(o.Quantity) AS UnitsSold, SUM(o.Quantity * m.Price) AS Revenue
            FROM Orders o JOIN MenuItems m ON m.ItemID = o.ItemID
            GROUP BY m.ItemID, m.ItemName, m.Category ORDER BY Revenue DESC;
        END""",
    "SalesByCategory": """
        CREATE PROCEDURE SalesByCategory()
        BEGIN
            SELECT m.Category, SUM(o.Quantity * m.Price) AS Revenue,
                   ROUND(100 * SUM(o.Quantity * m.Price) / (SELECT SUM(o2.Quantity * m2.Price) FROM Orders o2 JOIN MenuItems m2 ON m2.ItemID = o2.ItemID), 1) AS PctOfTotal
            FROM Orders o JOIN MenuItems m ON m.ItemID = o.ItemID
            GROUP BY m.Category ORDER BY Revenue DESC;
        END"""}
    for name, ddl in procs.items():
        cursor.execute(f"DROP PROCEDURE IF EXISTS {name}")
        cursor.execute(ddl)
        print(f"Stored procedure {name} is created.")
    connection.commit()
except Error as er:
    print("Error message:", er.msg)
finally:
    if connection is not None and connection.is_connected():
        cursor.close()
        connection.close()
        print("The connection is returned to the pool.")

A connection is taken from the pool.
Stored procedure GetBookingsWithStaff is created.
Stored procedure AddBooking is created.
Stored procedure UpdateBookingSlot is created.
Stored procedure CancelBooking is created.
Stored procedure SalesByItem is created.
Stored procedure SalesByCategory is created.
The connection is returned to the pool.


## Task 4: List the bookings with the assigned staff member

In [ ]:
def call_proc(name, args=()):
    """Take a connection from the pool, call a stored procedure, return its rows, then return the connection."""
    connection = pool.get_connection()
    try:
        cursor = connection.cursor()
        cursor.execute("USE little_lemon_db")
        cursor.callproc(name, args)
        rows = [r for res in cursor.stored_results() for r in res.fetchall()]
        connection.commit()
        return rows
    finally:
        cursor.close()
        connection.close()

try:
    print("A connection is taken from the pool.")
    for slot, guest, staff, role in call_proc("GetBookingsWithStaff"):
        print(f"BookingSlot {slot}")
        print(f"\tGuest_name: {guest}")
        print(f"\tAssigned to: {staff} [{role}]\n")
    print("The connection is returned to the pool.")
except Error as er:
    print("Error message:", er.msg)

A connection is taken from the pool.
BookingSlot 15:00:00
	Guest_name: Vanessa McCarthy
	Assigned to: Giorgos Dioudis [Head Chef]

BookingSlot 17:30:00
	Guest_name: Marcos Romero
	Assigned to: Fatma Kaya [Assistant Chef]

BookingSlot 18:00:00
	Guest_name: Anees Java
	Assigned to: John Millar [Receptionist]

BookingSlot 18:30:00
	Guest_name: Hiroki Yamane
	Assigned to: Elena Salvai [Head Waiter]

BookingSlot 19:00:00
	Guest_name: Anna Iversen
	Assigned to: Mario Gollini [Manager]

BookingSlot 19:00:00
	Guest_name: Joakim Iversen
	Assigned to: Adrian Gollini [Assistant Manager]

BookingSlot 20:00:00
	Guest_name: Diana Pinto
	Assigned to: John Millar [Receptionist]

The connection is returned to the pool.


## Task 5: Add a booking, change its slot, then cancel another

In [ ]:
try:
    new_id = call_proc("AddBooking", (10, "Sofia Marin", "21:00:00", 5))[0][0]
    print(f"Booking {new_id} for Sofia Marin is added and committed.")
    call_proc("UpdateBookingSlot", (new_id, "21:30:00"))
    print(f"Booking {new_id} is moved to 21:30:00 and committed.")
    call_proc("CancelBooking", (2,))
    print("Booking 2 (Joakim Iversen) is cancelled together with its orders, and committed.")
    print()
    for slot, guest, staff, role in call_proc("GetBookingsWithStaff"):
        print(f"{slot}  {guest:<18} -> {staff} [{role}]")
except Error as er:
    print("Error message:", er.msg)

Booking 8 for Sofia Marin is added and committed.
Booking 8 is moved to 21:30:00 and committed.
Booking 2 (Joakim Iversen) is cancelled together with its orders, and committed.

15:00:00  Vanessa McCarthy   -> Giorgos Dioudis [Head Chef]
17:30:00  Marcos Romero      -> Fatma Kaya [Assistant Chef]
18:00:00  Anees Java         -> John Millar [Receptionist]
18:30:00  Hiroki Yamane      -> Elena Salvai [Head Waiter]
19:00:00  Anna Iversen       -> Mario Gollini [Manager]
20:00:00  Diana Pinto        -> John Millar [Receptionist]
21:30:00  Sofia Marin        -> Elena Salvai [Head Waiter]


## Task 6: Sales report (SalesByItem and SalesByCategory)

In [ ]:
try:
    print("Sales by menu item")
    print(f"{'Item':<15}{'Category':<10}{'Units':>6}{'Revenue':>10}")
    for item, cat, units, rev in call_proc("SalesByItem"):
        print(f"{item:<15}{cat:<10}{int(units):>6}{float(rev):>10.2f}")
    print("\nSales by category")
    total = 0
    for cat, rev, pct in call_proc("SalesByCategory"):
        total += float(rev)
        print(f"{cat:<10}{float(rev):>10.2f}{float(pct):>7.1f}%")
    print(f"{'Total':<10}{total:>10.2f}")
except Error as er:
    print("Error message:", er.msg)

Sales by menu item
Item           Category   Units   Revenue
Grilled Fish   Mains          5    120.00
Pasta          Mains          6    111.00
Greek Salad    Starters       5     62.50
Lemon Dessert  Desserts       5     37.50
Greek Yoghurt  Desserts       3     18.00
Bruschetta     Starters       2     16.00

Sales by category
Mains         231.00   63.3%
Starters       78.50   21.5%
Desserts       55.50   15.2%
Total         365.00


## Task 7: Confirm the changes are permanent and the connections are back in the pool

In [14]:
try:
    connection = pool.get_connection()      # a pooled connection must see the committed data
    cursor = connection.cursor()
    cursor.execute("SELECT COUNT(*) FROM little_lemon_db.Bookings")
    print("Bookings visible from a pooled connection:", cursor.fetchone()[0])
    cursor.close()
    connection.close()
    c1, c2 = pool.get_connection(), pool.get_connection()   # both pool slots must be free again    
    print("Both pooled connections are available again, so none was left open.")
    c1.close()
    c2.close()
except Error as er:
    print("Error code:", er.errno)
    print("Error message:", er.msg)

Bookings visible from a pooled connection: 7
Both pooled connections are available again, so none was left open.
